In [1]:
from transformers import DonutProcessor, VisionEncoderDecoderModel
from PIL import Image
import torch

# Load the model and processor
processor = DonutProcessor.from_pretrained("chinmays18/medical-prescription-ocr")
model = VisionEncoderDecoderModel.from_pretrained("chinmays18/medical-prescription-ocr")

# Move to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Process an image
image = Image.open("prescription.png").convert("RGB")
pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device)

# Generate text
task_prompt = "<s_ocr>"
decoder_input_ids = processor.tokenizer(task_prompt, return_tensors="pt").input_ids.to(device)

generated_ids = model.generate(
    pixel_values,
    decoder_input_ids=decoder_input_ids,
    max_length=512,
    num_beams=1,
    early_stopping=True
)

# Decode the generated text
generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(generated_text)


The image processor of type `DonutImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


model.safetensors:   0%|          | 0.00/809M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Address West Rumb , Prince of the age-fefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefefeefficientistinglessness . He wasteefficiently-minuefficiently-minuefficiently- . He was assembling . He wasteefficiently- . He wasteectory- . He wasteectory- . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The . The 

In [2]:
pip install einops timm

   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   -------------------- ------------------- 1.3/2.6 MB 8.9 MB/s eta 0:00:01
   ---------------------------------------- 2.6/2.6 MB 10.6 MB/s  0:00:00

   ---------------------------------------- 0/2 [einops]
   -------------------- ------------------- 1/2 [timm]
   -------------------- ------------------- 1/2 [timm]
   -------------------- ------------------- 1/2 [timm]
   -------------------- ------------------- 1/2 [timm]
   -------------------- ------------------- 1/2 [timm]
   -------------------- ------------------- 1/2 [timm]
   -------------------- ------------------- 1/2 [timm]
   -------------------- ------------------- 1/2 [timm]
   -------------------- ------------------- 1/2 [timm]
   -------------------- ------------------- 1/2 [timm]
   -------------------- ------------------- 1/2 [timm]
   -------------------- ------------------- 1/2 [timm]
   -------------------- ------------------- 1/2 [timm]
   -----

In [1]:
import requests

import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM 


device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model = AutoModelForCausalLM.from_pretrained("microsoft/Florence-2-large", torch_dtype=torch_dtype, trust_remote_code=True,attn_implementation="eager",low_cpu_mem_usage=False).to(device)
processor = AutoProcessor.from_pretrained("microsoft/Florence-2-large", trust_remote_code=True,)

prompt = "Read the below prescription and extract all the medicines and their dosages in a JSON format. <OD>"
image = Image.open("prescription.png").convert("RGB")
image = image.resize((512, 512))
inputs = processor(text=prompt, images=image, return_tensors="pt").to(device, torch_dtype)

generated_ids = model.generate(
    input_ids=inputs["input_ids"],
    pixel_values=inputs["pixel_values"],
    max_new_tokens=4096,
    num_beams=3,
    do_sample=False
)
generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]

parsed_answer = processor.post_process_generation(generated_text, task="<OD>", image_size=(image.width, image.height))

print(parsed_answer)


`torch_dtype` is deprecated! Use `dtype` instead!


RuntimeError: Tensor.item() cannot be called on meta tensors